# ĐỒ ÁN MÔN BIG DATA: PHÂN TÍCH HÀNH VI KHÁCH HÀNG E-COMMERCE
## NOTEBOOK 02: HỌC MÁY PHÂN KHÚC KHÁCH HÀNG (K-MEANS & RFM) VÀ KHAI PHÁ GIỎ HÀNG (APRIORI)
---
- **Học phần:** Dữ liệu Lớn (Big Data Analytics)
- **Thuật toán áp dụng:** K-Means Clustering, Elbow Method, Silhouette Score, PCA, Apriori Algorithm
- **Mục tiêu:** Định danh 4 Chân dung Khách hàng và Khám phá Luật kết hợp Bán chéo (Cross-selling).

### 1. Nạp Dữ liệu Đặc trưng RFM cấp User từ BigQuery

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.bq_client import BigQueryService
from src.data_processor import CustomerDataProcessor
from src.clustering import CustomerClusterModel
from src.market_basket import MarketBasketAnalyzer

bq = BigQueryService()
query_rfm = bq.read_sql_file("03_rfm_features.sql")
df_rfm = bq.query_to_dataframe(query_rfm, cache_name="rfm_features")

print(f"✓ Nạp thành công {len(df_rfm):,} hồ sơ người dùng từ BigQuery!")
display(df_rfm.head(5))

### 2. Tiền xử lý Dữ liệu (IQR Clipping $\to$ Log1p Transform $\to$ StandardScaler)

In [2]:
processor = CustomerDataProcessor()
X_scaled, df_trans, df_clean = processor.fit_transform(df_rfm)

print("✓ Ma trận đặc trưng sau chuẩn hóa (StandardScaler):")
print("  • Mean:", np.mean(X_scaled, axis=0).round(4))
print("  • Std:", np.std(X_scaled, axis=0).round(4))
print("  • Shape:", X_scaled.shape)

### 3. Đánh giá Toán học Tìm $K$ Tối ưu (Phương pháp Elbow & Silhouette Score)

In [3]:
cluster_model = CustomerClusterModel(random_state=42)
eval_res = cluster_model.evaluate_k_range(X_scaled, k_min=2, k_max=8)

# Vẽ biểu đồ Elbow & Silhouette song song
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=eval_res["k_values"],
    y=eval_res["inertias"],
    mode="lines+markers",
    name="Inertia (Elbow Method)",
    line=dict(color="#3B82F6", width=3)
))
fig.add_trace(go.Scatter(
    x=eval_res["k_values"],
    y=eval_res["silhouette_scores"],
    mode="lines+markers",
    name="Silhouette Score",
    yaxis="y2",
    line=dict(color="#10B981", width=3)
))
fig.update_layout(
    title="Đánh giá Tìm số Cụm Tối ưu K (Elbow & Silhouette Curve)",
    xaxis=dict(title="Số lượng Cụm (K)"),
    yaxis=dict(title="Inertia (WCSS)"),
    yaxis2=dict(title="Silhouette Score", overlaying="y", side="right"),
    template="plotly_dark"
)
fig.show()

### 4. Huấn luyện K-Means ($K=4$) & Định danh 4 Chân dung Khách hàng (Personas)

In [4]:
df_segmented, cluster_summary = cluster_model.fit_predict(X_scaled, df_clean, n_clusters=4)
df_enriched, df_persona_table = cluster_model.profile_personas(df_segmented)

print("=== BẢNG TỔNG HỢP 4 CHÂN DUNG KHÁCH HÀNG (PERSONAS TABLE) ===")
display(df_persona_table)

#### Trực quan hóa Không gian Cụm 3D với PCA

In [5]:
fig_3d = px.scatter_3d(
    df_enriched.sample(min(500, len(df_enriched)), random_state=42),
    x="pca_3d_x",
    y="pca_3d_y",
    z="pca_3d_z",
    color="persona_name",
    hover_name="user_pseudo_id",
    title="Không gian Phân cụm Khách hàng 3D (PCA Projection)"
)
fig_3d.update_layout(template="plotly_dark")
fig_3d.show()

### 5. Khai phá Luật Kết hợp Giỏ hàng (Market Basket Analysis - Apriori)

In [6]:
query_market = bq.read_sql_file("04_market_basket.sql")
df_market = bq.query_to_dataframe(query_market, cache_name="market_basket")

mba = MarketBasketAnalyzer(min_support=0.04, min_confidence=0.25, min_lift=1.0)
basket_mat, n_valid = mba.build_basket_matrix(df_market)
freq_sets, rules = mba.run_apriori_and_rules(basket_mat)
df_recs = mba.get_cross_selling_recommendations(top_n=5)

print("=== BẢNG GỢI Ý SẢN PHẨM BÁN CHÉO (CROSS-SELLING RECOMMENDATIONS) ===")
display(df_recs)